[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/09_maximum_likelihood_and_map_estimation/first_principles.ipynb)

# Topic 09: Maximum Likelihood and MAP Estimation

## 1. First-Principles Intuition & Motivation

A probability model is a machine that turns parameters into data distributions. Inference runs the machine backwards: the data are given, the parameters are not. The central move of parametric statistics is to take the very same function $p(x \mid \theta)$ and *change which argument is held fixed*. With $\theta$ fixed and $x$ varying it is a density describing randomness; with $x$ fixed and $\theta$ varying it is the **likelihood**, a score-card ranking candidate explanations of what actually happened:

$$
L(\theta) = p(x_1, \ldots, x_n \mid \theta) = \prod_{i=1}^{n} p(x_i \mid \theta) \quad \text{(i.i.d. case)} .
$$

Maximum likelihood picks the parameter that assigns the highest probability to the data we saw. The rationale is not that this $\theta$ is "most probable" — likelihood is not a probability distribution over $\theta$ — but that it is the value under which the observed sample is least surprising.

### 1.1 Why the Logarithm Is Not a Convenience

Three independent reasons make $\ell(\theta) = \ln L(\theta)$ the working object.

1. **Products become sums.** $\ell(\theta) = \sum_{i=1}^n \ln p(x_i \mid \theta)$, so $\ell$ is an *average* (after dividing by $n$) and the law of large numbers applies directly. This is what makes the asymptotic theory possible.
2. **Numerical survival.** A product of $10^6$ densities underflows to $0$ in double precision; a sum of $10^6$ log-densities does not. Every likelihood computation in software is done in log space.
3. **Geometry.** $\ln$ is strictly increasing, so the maximizer is unchanged, but the log-likelihood of an exponential family is concave in the natural parameter — turning a general optimization into a convex one.

Dividing by $n$ makes the connection to information theory explicit:

$$
-\frac{1}{n}\ell(\theta) = -\frac{1}{n}\sum_{i=1}^n \ln p(x_i \mid \theta) \xrightarrow[n\to\infty]{\text{a.s.}} -E_{p_0}\left[\ln p(X \mid \theta)\right] = H(p_0) + D_{\text{KL}}\left(p_0 \,\Vert\, p_\theta\right) .
$$

The left side is the average negative log-likelihood — the cross-entropy loss of machine learning. The right side is a constant plus a KL divergence. Maximizing likelihood *is* minimizing KL divergence to the truth.

### 1.2 Curvature Is Information

Two datasets can select the same $\hat\theta$ yet differ enormously in how confident that choice is. The distinguishing feature is the *curvature* of the log-likelihood at its peak: a sharp spike means nearby parameters explain the data much worse, a flat ridge means they explain it nearly as well. Quantifying that curvature gives the Fisher information

$$
I(\theta) = -E\left[\frac{\partial^2}{\partial\theta^2}\ln p(X\mid\theta)\right] ,
$$

and the asymptotic variance of the MLE is its inverse, $\operatorname{Var}(\hat\theta_n) \approx 1/\left(nI(\theta)\right)$. Everything downstream — standard errors, confidence intervals, Wald tests, natural-gradient preconditioners, Laplace approximations to posteriors — is a use of this one identity.

### 1.3 Priors Enter as Regularizers

When data are scarce the likelihood can be maximized at absurd values: three coin flips landing heads give $\hat p = 1$, asserting that tails is impossible. Prior information fixes this. Bayes' rule gives

$$
p(\theta \mid x) = \frac{p(x \mid \theta)\,p(\theta)}{p(x)} \ \propto\ p(x\mid\theta)\,p(\theta) ,
$$

and taking logs turns the product into a sum:

$$
\hat\theta_{\text{MAP}} = \arg\max_\theta \Big[\underbrace{\ell(\theta)}_{\text{fit}} + \underbrace{\ln p(\theta)}_{-\text{penalty}}\Big] .
$$

Because $\ell$ grows like $n$ while $\ln p(\theta)$ does not, the prior's influence decays as $O(1/n)$: it dominates when data are few and vanishes when data are plentiful. This single line explains why weight decay helps small models more than large-data regimes, and why smoothing constants matter for rare $n$-grams but not for frequent ones.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Likelihood and log-likelihood).** For a statistical model $\{p(\cdot \mid \theta) : \theta \in \Theta \subseteq \mathbb{R}^d\}$ and observed data $x = (x_1,\ldots,x_n)$,

$$
L(\theta) = p(x \mid \theta), \qquad \ell(\theta) = \ln L(\theta) = \sum_{i=1}^n \ln p(x_i \mid \theta) \ \ \text{(i.i.d.)} .
$$

Likelihoods are defined up to a positive multiplicative constant not depending on $\theta$; only ratios carry meaning.

**Definition 2.2 (Maximum likelihood estimator).**

$$
\hat\theta_{\text{MLE}} \in \arg\max_{\theta \in \Theta} \ell(\theta) .
$$

At an interior maximum with differentiable $\ell$, $\hat\theta$ solves the *score equation* $\nabla_\theta \ell(\hat\theta) = 0$; boundary maxima (e.g. $\text{Unif}(0,\theta)$) must be found by direct inspection.

**Theorem 2.3 (Invariance).** If $\hat\theta$ is an MLE of $\theta$ and $g$ is any function, then $g(\hat\theta)$ is an MLE of $g(\theta)$. (Proof via the induced profile likelihood $L^*(\eta) = \sup\{L(\theta) : g(\theta) = \eta\}$.)

**Definition 2.4 (Score function).**

$$
s(\theta; x) = \nabla_\theta \ln p(x \mid \theta) .
$$

**Definition 2.5 (Fisher information).** For a single observation,

$$
I(\theta) = E_\theta\left[s(\theta; X)\,s(\theta; X)^{\top}\right] .
$$

For $n$ i.i.d. observations the total information is $nI(\theta)$ by additivity.

**Theorem 2.6 (Score identities).** Under regularity conditions permitting differentiation under the integral sign (support not depending on $\theta$, $p$ smooth in $\theta$):

$$
E_\theta\left[s(\theta; X)\right] = 0, \qquad I(\theta) = \operatorname{Cov}_\theta\left(s(\theta;X)\right) = -E_\theta\left[\nabla_\theta^2 \ln p(X\mid\theta)\right] .
$$

The last equality is the **information equality**: expected curvature equals score variance.

**Theorem 2.7 (Cramér–Rao lower bound).** Let $T(X)$ be an unbiased estimator of $\theta$ (scalar case) in a regular model. Then

$$
\operatorname{Var}_\theta\left(T\right) \ \ge\ \frac{1}{n\,I(\theta)} .
$$

More generally, for an estimator of $g(\theta)$ with $E_\theta[T] = g(\theta)$, $\operatorname{Var}(T) \ge \left[g'(\theta)\right]^2 / \left(nI(\theta)\right)$. An estimator attaining the bound is called *efficient*.

**Definition 2.8 (Consistency).** $\hat\theta_n$ is consistent if $\hat\theta_n \xrightarrow{P} \theta_0$ under the true parameter $\theta_0$.

**Theorem 2.9 (Consistency of the MLE).** If the model is identifiable, $\Theta$ compact, $\ln p(x\mid\theta)$ continuous in $\theta$ and dominated by an integrable envelope, then $\hat\theta_n \xrightarrow{P}\theta_0$.

**Theorem 2.10 (Asymptotic normality and efficiency).** Under regularity (interior $\theta_0$, twice differentiability, nonsingular $I(\theta_0)$, uniform LLN for the Hessian),

$$
\sqrt{n}\left(\hat\theta_n - \theta_0\right) \xrightarrow{d} \mathcal{N}\left(0, \ I(\theta_0)^{-1}\right) .
$$

Hence the MLE is asymptotically unbiased, asymptotically normal, and asymptotically efficient: it attains the Cramér–Rao bound in the limit.

**Definition 2.11 (MAP estimator).** Given a prior density $p(\theta)$,

$$
\hat\theta_{\text{MAP}} = \arg\max_{\theta} p(\theta \mid x) = \arg\max_\theta \left[\ell(\theta) + \ln p(\theta)\right] .
$$

**Proposition 2.12 (Prior–penalty dictionary).**

$$
\theta \sim \mathcal{N}(0, \tau^2 I) \ \Leftrightarrow\ \text{penalty } \frac{1}{2\tau^2}\Vert\theta\Vert_2^2 \ (\text{ridge}); \qquad \theta_j \sim \text{Laplace}(0,b) \ \Leftrightarrow\ \text{penalty } \frac{1}{b}\Vert\theta\Vert_1 \ (\text{lasso}) .
$$

**Definition 2.13 (Exponential family).** A family is exponential in canonical form if

$$
p(x \mid \eta) = h(x)\exp\left(\eta^{\top} T(x) - A(\eta)\right) ,
$$

with sufficient statistic $T$, natural parameter $\eta$, and log-partition $A$. Then $\nabla A(\eta) = E[T(X)]$, $\nabla^2 A(\eta) = \operatorname{Cov}(T(X)) = I(\eta)$, so $A$ is convex and the log-likelihood is concave. The MLE solves the **moment-matching** equation

$$
\nabla A(\hat\eta) = \frac{1}{n}\sum_{i=1}^n T(x_i) .
$$

**Theorem 2.14 (Fisher–Neyman factorization).** $T(X)$ is sufficient for $\theta$ iff $p(x\mid\theta) = g\left(T(x), \theta\right)h(x)$. The MLE depends on the data only through a sufficient statistic.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: Closed-Form MLEs — Bernoulli, Gaussian, Poisson, Uniform

**(a) Bernoulli($p$).** With $k = \sum_i x_i$ successes in $n$ trials,

$$
\ell(p) = k\ln p + (n-k)\ln(1-p), \qquad \ell'(p) = \frac{k}{p} - \frac{n-k}{1-p} .
$$

Setting $\ell' = 0$ gives $k(1-p) = (n-k)p$, hence $k = np$ and

$$
\hat p = \frac{k}{n} = \bar{x} .
$$

The second derivative $\ell''(p) = -k/p^2 - (n-k)/(1-p)^2 \lt 0$ confirms a maximum.

**(b) Gaussian $\mathcal{N}(\mu, \sigma^2)$, both unknown.**

$$
\ell(\mu,\sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln\sigma^2 - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i-\mu)^2 .
$$

Differentiating in $\mu$: $\frac{1}{\sigma^2}\sum_i (x_i - \mu) = 0 \Rightarrow \hat\mu = \bar{x}$. Differentiating in $v = \sigma^2$:

$$
\frac{\partial \ell}{\partial v} = -\frac{n}{2v} + \frac{1}{2v^2}\sum_i (x_i-\hat\mu)^2 = 0 \quad \Longrightarrow \quad \hat\sigma^2 = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^2 .
$$

Note the divisor $n$, not $n-1$: since $E[\hat\sigma^2] = \frac{n-1}{n}\sigma^2$, the MLE is biased low. Bias is unavoidable if invariance is desired — $\widehat{\sigma} = \sqrt{\hat\sigma^2}$ must follow from $\hat\sigma^2$ by Theorem 2.3, and no single convention can debias every reparameterization.

**(c) Poisson($\lambda$).** $\ell(\lambda) = -n\lambda + \left(\sum_i x_i\right)\ln\lambda - \sum_i \ln(x_i!)$, so $\ell'(\lambda) = -n + \frac{\sum_i x_i}{\lambda} = 0$ gives $\hat\lambda = \bar{x}$.

**(d) Uniform $\text{Unif}(0,\theta)$ — a boundary case.** $L(\theta) = \theta^{-n}\mathbf{1}\{\theta \ge \max_i x_i\}$ is *decreasing* wherever it is positive, so calculus never applies; the maximum sits at the constraint boundary:

$$
\hat\theta = \max_i x_i = x_{(n)} .
$$

This model violates the regularity conditions (support depends on $\theta$), and indeed $\hat\theta$ converges at rate $n$, not $\sqrt{n}$, with an exponential rather than normal limit — a reminder that the asymptotic theory has genuine hypotheses. $\blacksquare$

### Proof 3.2: The Score Has Mean Zero and the Information Equality

**Claim.** Under regularity, $E_\theta[s(\theta;X)] = 0$ and $I(\theta) = \operatorname{Cov}(s) = -E\left[\nabla^2_\theta \ln p(X\mid\theta)\right]$.

**Proof (mean zero).** Start from the normalization $\int p(x\mid\theta)\,dx = 1$ for every $\theta$ and differentiate both sides in $\theta$. Regularity licenses exchanging derivative and integral:

$$
0 = \nabla_\theta \int p(x\mid\theta)\,dx = \int \nabla_\theta p(x\mid\theta)\,dx .
$$

Now use the log-derivative ("REINFORCE") identity $\nabla_\theta p = p\,\nabla_\theta \ln p$:

$$
0 = \int p(x\mid\theta)\,\nabla_\theta \ln p(x\mid\theta)\,dx = E_\theta\left[s(\theta;X)\right] . \qquad \square
$$

Consequently $I(\theta) = E[ss^{\top}] = \operatorname{Cov}(s)$ with no centering correction.

**Proof (information equality).** Differentiate the identity $\int p\,\nabla_\theta \ln p \,dx = 0$ once more:

$$
0 = \int \nabla_\theta\left[p\,\nabla_\theta \ln p\right] dx = \int \left(\nabla_\theta p\right)\left(\nabla_\theta \ln p\right)^{\top} dx + \int p\,\nabla^2_\theta \ln p\ dx .
$$

Apply the log-derivative identity to the first term again, $\nabla_\theta p = p\nabla_\theta \ln p$:

$$
0 = E\left[\left(\nabla_\theta \ln p\right)\left(\nabla_\theta\ln p\right)^{\top}\right] + E\left[\nabla^2_\theta \ln p\right] = I(\theta) + E\left[\nabla^2_\theta \ln p\right] . \qquad \blacksquare
$$

**Reading.** Information is measured equivalently by how much the score fluctuates (first-order sensitivity) or by how sharply the log-likelihood curves down (second-order geometry). The equality fails when the model is misspecified, and the discrepancy produces the sandwich ("robust", Huber–White) covariance $I^{-1}JI^{-1}$.

**Worked example (Bernoulli).** $\ln p(x\mid p) = x\ln p + (1-x)\ln(1-p)$, so $s = \frac{x}{p} - \frac{1-x}{1-p} = \frac{x - p}{p(1-p)}$. Then

$$
I(p) = \operatorname{Var}\left(\frac{X-p}{p(1-p)}\right) = \frac{p(1-p)}{p^2(1-p)^2} = \frac{1}{p(1-p)} ,
$$

which blows up as $p \to 0$ or $1$: extreme probabilities are *easy* to pin down on the natural scale.

### Proof 3.3: The Cramér–Rao Lower Bound

**Claim.** For an unbiased estimator $T = T(X_1,\ldots,X_n)$ of $\theta$ in a regular model, $\operatorname{Var}(T) \ge 1/\left(nI(\theta)\right)$.

**Proof.** Write $s_n(\theta) = \sum_{i=1}^n s(\theta; X_i)$ for the total score, which has mean $0$ and variance $nI(\theta)$ by independence.

**Step 1 — covariance identity.** Differentiate the unbiasedness relation $E_\theta[T] = \int T(x)p(x\mid\theta)dx = \theta$ with respect to $\theta$ and exchange:

$$
1 = \int T(x)\,\nabla_\theta p(x\mid\theta)\,dx = \int T(x)\,s_n(\theta;x)\,p(x\mid\theta)\,dx = E\left[T\,s_n\right] .
$$

Since $E[s_n] = 0$, this says $\operatorname{Cov}(T, s_n) = 1$.

**Step 2 — Cauchy–Schwarz.** For any two square-integrable random variables,

$$
\operatorname{Cov}(T, s_n)^2 \le \operatorname{Var}(T)\operatorname{Var}(s_n) .
$$

Substituting $\operatorname{Cov} = 1$ and $\operatorname{Var}(s_n) = nI(\theta)$:

$$
1 \le \operatorname{Var}(T)\cdot nI(\theta) \quad \Longrightarrow \quad \operatorname{Var}(T) \ge \frac{1}{nI(\theta)} . \qquad \blacksquare
$$

**Equality condition.** Cauchy–Schwarz is tight iff $T - \theta = c(\theta)\,s_n(\theta)$ almost surely, i.e. the score is an affine function of the estimator. This happens exactly in exponential families with $T$ the natural sufficient statistic — which is why $\bar{X}$ is efficient for the Gaussian mean and the Poisson rate but no unbiased estimator is efficient for, say, $e^{-\lambda}$ at finite $n$.

**Multivariate form.** $\operatorname{Cov}(T) \succeq \left[nI(\theta)\right]^{-1}$ in the Loewner order; the same proof runs with a matrix Cauchy–Schwarz.

**Caveat.** Unbiasedness is essential. Ridge regression, James–Stein shrinkage, and MAP estimates are all biased and routinely achieve smaller mean squared error than any unbiased estimator, since $\text{MSE} = \text{bias}^2 + \text{variance}$ permits trading one for the other.

### Proof 3.4: Consistency of the MLE via KL Divergence

**Claim.** If the model is identifiable and regularity conditions hold, $\hat\theta_n \xrightarrow{P}\theta_0$.

**Proof.** Define the normalized log-likelihood ratio

$$
M_n(\theta) = \frac{1}{n}\sum_{i=1}^n \ln \frac{p(X_i\mid\theta)}{p(X_i \mid \theta_0)} .
$$

Maximizing $M_n$ is equivalent to maximizing $\ell$, since the denominator does not involve $\theta$.

**Step 1 — pointwise limit.** By the strong law of large numbers, for each fixed $\theta$,

$$
M_n(\theta) \xrightarrow{a.s.} M(\theta) = E_{\theta_0}\left[\ln\frac{p(X\mid\theta)}{p(X\mid\theta_0)}\right] = -D_{\text{KL}}\left(p_{\theta_0}\,\Vert\,p_\theta\right) .
$$

**Step 2 — the limit is uniquely maximized at $\theta_0$.** By Jensen's inequality applied to the concave $\ln$,

$$
M(\theta) = E_{\theta_0}\left[\ln \frac{p(X\mid\theta)}{p(X\mid\theta_0)}\right] \le \ln E_{\theta_0}\left[\frac{p(X\mid\theta)}{p(X\mid\theta_0)}\right] = \ln\int p(x\mid\theta)\,dx = \ln 1 = 0 ,
$$

with equality iff the ratio is constant almost surely, i.e. $p_\theta = p_{\theta_0}$, which identifiability upgrades to $\theta = \theta_0$. So $M(\theta_0) = 0 \gt M(\theta)$ for $\theta \ne \theta_0$.

**Step 3 — uniformity transfers the maximizer.** With $\Theta$ compact and $\ln p(x\mid\theta)$ continuous and dominated, the convergence in Step 1 is uniform: $\sup_{\theta\in\Theta}\vert M_n(\theta) - M(\theta)\vert \xrightarrow{P} 0$. Standard argmax continuity then gives $\hat\theta_n \xrightarrow{P}\theta_0$: for any neighborhood $U$ of $\theta_0$, $\sup_{\theta \notin U} M(\theta) = -\delta \lt 0$, and once the uniform error is below $\delta/2$ the maximizer of $M_n$ must lie in $U$. $\blacksquare$

**Interpretation.** Consistency of the MLE is the statement that the empirical KL divergence to the model converges to the true KL divergence, and that KL separates distinct distributions. Under misspecification ($p_0$ outside the family) the same argument shows $\hat\theta_n \to \theta^\star = \arg\min_\theta D_{\text{KL}}(p_0\Vert p_\theta)$ — the MLE finds the KL-closest member, which is precisely what a neural network trained with cross-entropy does.

### Proof 3.5: Asymptotic Normality of the MLE

**Claim.** $\sqrt{n}\left(\hat\theta_n - \theta_0\right) \xrightarrow{d} \mathcal{N}\left(0, I(\theta_0)^{-1}\right)$.

**Proof.** Let $\ell_n(\theta) = \sum_i \ln p(X_i\mid\theta)$ and expand the score at $\hat\theta_n$ around $\theta_0$ using the mean value theorem componentwise:

$$
0 = \nabla \ell_n(\hat\theta_n) = \nabla\ell_n(\theta_0) + \nabla^2\ell_n(\tilde\theta_n)\left(\hat\theta_n - \theta_0\right), \qquad \tilde\theta_n \text{ between } \hat\theta_n \text{ and } \theta_0 .
$$

Solve for the estimation error and insert $\sqrt{n}$ factors in matching places:

$$
\sqrt{n}\left(\hat\theta_n - \theta_0\right) = \left[-\frac{1}{n}\nabla^2 \ell_n(\tilde\theta_n)\right]^{-1}\left[\frac{1}{\sqrt{n}}\nabla\ell_n(\theta_0)\right] .
$$

**Numerator — a CLT.** $\frac{1}{\sqrt n}\nabla\ell_n(\theta_0) = \sqrt{n}\cdot\frac{1}{n}\sum_i s(\theta_0; X_i)$ is a standardized sum of i.i.d. scores with mean $0$ (Proof 3.2) and covariance $I(\theta_0)$. By the central limit theorem,

$$
\frac{1}{\sqrt n}\nabla\ell_n(\theta_0)\ \xrightarrow{d}\ \mathcal{N}\left(0, I(\theta_0)\right) .
$$

**Denominator — an LLN.** $-\frac1n\nabla^2\ell_n(\theta_0) = -\frac1n\sum_i \nabla^2 \ln p(X_i\mid\theta_0) \xrightarrow{P} -E\left[\nabla^2\ln p\right] = I(\theta_0)$ by the law of large numbers and the information equality. Consistency ($\tilde\theta_n \xrightarrow{P}\theta_0$, squeezed by $\hat\theta_n$) plus a uniform LLN transfers this to the evaluation point $\tilde\theta_n$.

**Combine with Slutsky.** A product of a term converging in probability to the constant matrix $I(\theta_0)^{-1}$ and a term converging in distribution gives

$$
\sqrt{n}\left(\hat\theta_n-\theta_0\right)\ \xrightarrow{d}\ I(\theta_0)^{-1}\mathcal{N}\left(0, I(\theta_0)\right) = \mathcal{N}\left(0,\ I(\theta_0)^{-1}I(\theta_0)I(\theta_0)^{-1}\right) = \mathcal{N}\left(0, I(\theta_0)^{-1}\right) . \qquad \blacksquare
$$

**Practical payoff.** Standard errors are read off the observed information $\hat{J} = -\nabla^2\ell_n(\hat\theta_n)$:

$$
\widehat{\operatorname{se}}(\hat\theta_j) = \sqrt{\left[\hat{J}^{-1}\right]_{jj}}, \qquad \text{Wald CI: } \hat\theta_j \pm z_{1-\alpha/2}\,\widehat{\operatorname{se}}(\hat\theta_j) .
$$

**Under misspecification.** If the true law is not in the family, the numerator's covariance $J = \operatorname{Cov}(s)$ no longer equals the denominator's limit $H = -E[\nabla^2\ln p]$, and the limit becomes the **sandwich** $\mathcal{N}(0, H^{-1}JH^{-1})$ — the theoretical basis of robust standard errors.

### Proof 3.6: MAP Estimation Is Penalized Likelihood

**Claim.** Gaussian priors give ridge regression; Laplace priors give lasso; Beta and Dirichlet priors give additive smoothing.

**(a) Ridge from a Gaussian prior.** Take $y_i = x_i^{\top}w + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$ and prior $w \sim \mathcal{N}(0, \tau^2 I_d)$. Then

$$
\ln p(w \mid y) = -\frac{1}{2\sigma^2}\sum_{i=1}^n\left(y_i - x_i^{\top}w\right)^2 - \frac{1}{2\tau^2}\Vert w\Vert_2^2 + \text{const} .
$$

Multiplying by $-2\sigma^2$ turns maximization into the ridge objective

$$
\hat{w}_{\text{MAP}} = \arg\min_w \left\{\Vert y - Xw\Vert_2^2 + \lambda\Vert w\Vert_2^2\right\}, \qquad \lambda = \frac{\sigma^2}{\tau^2} ,
$$

with the closed form obtained by setting the gradient $-2X^{\top}(y - Xw) + 2\lambda w$ to zero:

$$
\hat{w}_{\text{MAP}} = \left(X^{\top}X + \lambda I\right)^{-1}X^{\top}y .
$$

The regularization strength is the *noise-to-prior variance ratio*: a confident prior (small $\tau$) or noisy data (large $\sigma$) both shrink more, and $\lambda \to 0$ recovers the MLE.

**(b) Lasso from a Laplace prior.** With $p(w_j) = \frac{1}{2b}e^{-\vert w_j\vert/b}$,

$$
-\ln p(w) = \frac{1}{b}\sum_j \vert w_j\vert + \text{const} \quad \Longrightarrow \quad \hat w_{\text{MAP}} = \arg\min_w\left\{\Vert y - Xw\Vert_2^2 + \lambda\Vert w\Vert_1\right\},\ \lambda = \frac{2\sigma^2}{b} .
$$

The Laplace density has a kink at $0$, giving a non-differentiable penalty whose subgradient condition can be satisfied exactly at $w_j = 0$ — this is why lasso produces sparse solutions and ridge does not.

**(c) Additive smoothing from a Beta prior.** For $k$ successes in $n$ Bernoulli trials with $p \sim \text{Beta}(\alpha,\beta)$,

$$
\ln p(p\mid x) = (k + \alpha - 1)\ln p + (n - k + \beta - 1)\ln(1-p) + \text{const} .
$$

Differentiating and setting to zero, the same algebra as Proof 3.1(a) with shifted counts gives

$$
\hat p_{\text{MAP}} = \frac{k + \alpha - 1}{n + \alpha + \beta - 2} .
$$

With $\alpha = \beta = 2$ this is Laplace's rule of succession $\frac{k+1}{n+2}$: three heads out of three now yields $4/5$ rather than the MLE's absurd $1$. The prior acts as $\alpha - 1$ pseudo-successes and $\beta-1$ pseudo-failures, and its weight is $O(1)$ against $n$ real observations — hence $\hat p_{\text{MAP}} - \hat p_{\text{MLE}} = O(1/n)$. $\blacksquare$

**A warning about MAP.** The mode is not reparameterization-equivariant: if $\phi = g(\theta)$, the posterior density of $\phi$ carries a Jacobian factor $\left\vert dg^{-1}/d\phi\right\vert$, so $\arg\max_\phi p(\phi\mid x) \ne g\left(\arg\max_\theta p(\theta\mid x)\right)$ in general. The posterior *mean* and the full posterior are equivariant in the appropriate sense; MLE is equivariant by Theorem 2.3. MAP occupies an awkward middle ground — best understood as regularized optimization rather than as Bayesian inference.

### Proof 3.7: Expectation–Maximization for Latent-Variable Likelihoods

**Setting.** With latent variables $z$, the marginal likelihood $\ell(\theta) = \sum_i \ln \int p(x_i, z\mid\theta)\,dz$ has no closed form. Introduce any distribution $q(z)$ and apply Jensen:

$$
\ln p(x\mid\theta) = \ln \int q(z)\frac{p(x,z\mid\theta)}{q(z)}dz \ \ge\ \int q(z)\ln\frac{p(x,z\mid\theta)}{q(z)}dz =: \mathcal{F}(q,\theta) .
$$

**Exact gap.** Subtracting,

$$
\ln p(x\mid\theta) - \mathcal{F}(q,\theta) = D_{\text{KL}}\left(q(z)\,\Vert\,p(z\mid x,\theta)\right) \ \ge\ 0 ,
$$

so the bound is tight exactly when $q = p(z\mid x,\theta)$.

**The algorithm.** Alternate:

- **E-step**: $q^{(t)}(z) = p(z\mid x,\theta^{(t)})$, which makes the gap zero, so $\mathcal{F}(q^{(t)},\theta^{(t)}) = \ell(\theta^{(t)})$.
- **M-step**: $\theta^{(t+1)} = \arg\max_\theta E_{q^{(t)}}\left[\ln p(x,z\mid\theta)\right]$, which increases $\mathcal{F}$.

**Monotonicity proof.** Chaining the two steps,

$$
\ell(\theta^{(t+1)}) \ \ge\ \mathcal{F}\left(q^{(t)},\theta^{(t+1)}\right)\ \ge\ \mathcal{F}\left(q^{(t)},\theta^{(t)}\right) = \ell\left(\theta^{(t)}\right) ,
$$

using the bound, the M-step's optimality, and E-step tightness in turn. The likelihood never decreases. $\blacksquare$

**Consequences.** EM converges to a stationary point (not necessarily global — Gaussian mixtures have many local maxima and an unbounded likelihood as a component variance collapses onto a single data point). The same bound $\mathcal{F}$, with $q$ restricted to a tractable family instead of set to the exact posterior, is the **ELBO** of variational inference and the training objective of the variational autoencoder.

## 4. Computational & Algorithmic Insights

### 4.1 Solving the Score Equation

Closed-form MLEs exist only for simple exponential families. In general one solves $\nabla\ell(\theta) = 0$ numerically:

- **Newton–Raphson**: $\theta_{t+1} = \theta_t + \left[-\nabla^2\ell(\theta_t)\right]^{-1}\nabla\ell(\theta_t)$. Quadratic convergence near the optimum; cost $O(d^3)$ per step and requires the Hessian to be negative definite.
- **Fisher scoring**: replace the observed Hessian by $-nI(\theta_t)$. The expected information is always positive semi-definite, so the step is always an ascent direction; for generalized linear models this is exactly **IRLS** (iteratively reweighted least squares).
- **Natural gradient**: $\theta_{t+1} = \theta_t + \eta\,I(\theta_t)^{-1}\nabla\ell$ — Fisher scoring viewed as steepest ascent in the KL geometry rather than the Euclidean one; reparameterization-invariant, and the ancestor of K-FAC and Adam-style preconditioning.
- **Stochastic gradient**: for large $n$, $\nabla\ell$ is a sum, so minibatch estimates are unbiased; this is neural-network training.
- **EM**: for latent-variable models where the complete-data likelihood is tractable but the marginal is not.

### 4.2 Numerical Hygiene

- **Always work in log space.** Use `logsumexp` for mixture responsibilities and softmax normalizers: $\ln\sum_k e^{a_k} = a^\star + \ln\sum_k e^{a_k - a^\star}$ with $a^\star = \max_k a_k$ avoids both overflow and underflow.
- **Reparameterize constrained parameters.** Optimize $\ln\sigma$ instead of $\sigma \gt 0$, and logits instead of simplex probabilities; the optimizer then sees an unconstrained, better-conditioned problem, and the invariance property (Theorem 2.3) guarantees the answer maps back correctly.
- **Watch for unbounded likelihoods.** Gaussian mixtures let $\ell \to \infty$ as one component's variance collapses onto a data point; variance floors or a mild inverse-gamma prior (i.e. MAP rather than MLE) removes the degeneracy.
- **Separation in logistic regression.** If classes are linearly separable the MLE diverges, $\Vert\hat w\Vert \to \infty$; any proper prior (ridge penalty) restores a finite maximizer. This is the single most common "training diverges" pathology with a purely statistical explanation.
- **Standard errors from the Hessian.** Use the observed information at the optimum, and check it is well-conditioned; a near-singular Hessian signals a flat ridge, i.e. weak or non-identifiability, not a numerical accident.

### 4.3 Model Selection and the Laplace Approximation

A second-order expansion of the log-posterior around $\hat\theta_{\text{MAP}}$ gives the **Laplace approximation**

$$
p(\theta\mid x) \approx \mathcal{N}\left(\hat\theta_{\text{MAP}},\ \left[-\nabla^2\ln p(\theta\mid x)\big|_{\hat\theta}\right]^{-1}\right) ,
$$

and integrating that Gaussian yields an approximation to the marginal likelihood whose leading terms are the **BIC**:

$$
\ln p(x) \approx \ell(\hat\theta) - \frac{d}{2}\ln n .
$$

The companion **AIC** $= 2d - 2\ell(\hat\theta)$ arises instead from an asymptotically unbiased estimate of out-of-sample expected log-likelihood. Both penalize dimension because unpenalized likelihood always prefers the larger model: adding parameters can never decrease the maximum. Modern practice uses held-out log-likelihood, cross-validation, or WAIC, but all rest on the same observation that in-sample likelihood is an optimistically biased estimate of predictive performance.

## 5. Real-World Physics & AI/ML Applications

### 5.1 AI / Machine Learning

- **Cross-entropy loss is negative log-likelihood.** For a classifier with softmax outputs $p_\theta(y \mid x)$, minimizing $-\frac1n\sum_i \ln p_\theta(y_i\mid x_i)$ is exactly MLE for a categorical observation model. Squared-error regression is MLE under Gaussian noise; mean-absolute-error regression is MLE under Laplace noise; Poisson-deviance losses correspond to count models.
- **Weight decay is a Gaussian prior.** A penalty $\lambda\Vert w\Vert_2^2$ corresponds to $w \sim \mathcal{N}(0, \sigma^2/(2\lambda))$; label smoothing corresponds to a Dirichlet prior on the output distribution; dropout has an approximate variational-Bayes reading.
- **Logistic regression and GLMs.** The score equation $X^{\top}(y - \mu(\theta)) = 0$ is moment matching in an exponential family, solved by IRLS/Fisher scoring; its Hessian is the Fisher information used for Wald confidence intervals on coefficients.
- **Uncertainty in deep learning.** The Laplace approximation around a trained network's weights, using a Kronecker-factored Fisher, produces calibrated posteriors cheaply; the same Fisher powers natural-gradient and K-FAC optimizers.
- **Latent-variable generative models.** EM fits Gaussian mixtures and HMMs (Baum–Welch); its variational relaxation trains VAEs by maximizing the ELBO; contrastive objectives such as InfoNCE are likelihood ratios in disguise.
- **Language models.** Next-token training is maximum likelihood over a categorical distribution; perplexity $= \exp(-\ell/n)$ is a monotone transform of the achieved log-likelihood, and additive $n$-gram smoothing is Dirichlet-prior MAP.

### 5.2 Physics & Engineering

- **Particle physics.** Signal and background yields are fit by extended maximum likelihood over Poisson counts; profile-likelihood ratios provide the test statistic behind discovery claims (Wilks' theorem gives $-2\ln\Lambda \xrightarrow{d}\chi^2_k$), and nuisance parameters are handled by profiling or marginalization.
- **Radiometric dating and counting experiments.** Decay-rate estimation from event counts is Poisson MLE, $\hat\lambda = \bar{x}$, with Fisher information $I(\lambda) = 1/\lambda$ giving the familiar $\sqrt{N}$ counting error.
- **Sensor fusion and Kalman filtering.** The Kalman update is the MAP estimate of the state under Gaussian prior (prediction) and Gaussian likelihood (measurement); the gain matrix is exactly the precision-weighted combination of the two information matrices.
- **Spectral fitting and inverse problems.** Fitting line shapes, tomography, and deconvolution are ill-posed MLEs; Tikhonov regularization is a Gaussian prior, and total-variation regularization is a Laplace-type prior on gradients.
- **Design of experiments.** Optimal experimental design maximizes a scalar function of the Fisher information matrix (D-, A-, E-optimality) — choosing measurement conditions so that the log-likelihood curves as sharply as possible in the directions of interest.

### 5.3 Key Formula Summary

| Object | Formula | Notes |
|---|---|---|
| Log-likelihood | $\ell(\theta) = \sum_i \ln p(x_i \mid \theta)$ | Sum, not product; enables LLN/CLT |
| Score | $s(\theta) = \nabla_\theta \ln p(x \mid \theta)$ | $E[s] = 0$ under regularity |
| Fisher information | $I(\theta) = \operatorname{Cov}(s) = -E[\nabla^2 \ln p]$ | Information equality |
| MLE | $\nabla \ell(\hat\theta) = 0$ | Check boundaries when support depends on $\theta$ |
| Cramér–Rao | $\operatorname{Var}(T) \ge 1/(nI(\theta))$ | Unbiased estimators only |
| Asymptotics | $\sqrt{n}(\hat\theta - \theta_0) \xrightarrow{d} \mathcal{N}(0, I^{-1})$ | Efficient in the limit |
| MAP | $\hat\theta = \arg\max [\ell(\theta) + \ln p(\theta)]$ | Loss plus regularizer |
| Ridge | $\hat{w} = (X^{\top}X + \lambda I)^{-1}X^{\top}y$, $\lambda = \sigma^2/\tau^2$ | Gaussian prior |
| Beta-MAP | $\hat p = \frac{k+\alpha-1}{n+\alpha+\beta-2}$ | Additive smoothing |
| Exponential family MLE | $\nabla A(\hat\eta) = \frac1n\sum_i T(x_i)$ | Moment matching |
| EM bound | $\ln p(x \mid \theta) = \mathcal{F}(q,\theta) + D_{\text{KL}}(q \Vert p(z \mid x,\theta))$ | ELBO; tight at the exact posterior |

## 6. Canonical Literature Mapping & References

| Source | Chapters / Sections | Coverage |
|---|---|---|
| Casella & Berger, *Statistical Inference* (2nd ed.) | Chapters 7, 10 | Point estimation, Cramér–Rao, asymptotic efficiency. |
| Wasserman, *All of Statistics* | Chapters 9, 11 | Compact MLE theory, delta method, Bayesian comparison. |
| Lehmann & Casella, *Theory of Point Estimation* (2nd ed.) | Chapters 2, 6 | Sufficiency, completeness, asymptotic optimality. |
| van der Vaart, *Asymptotic Statistics* | Chapters 5, 8 | M-estimators, LAN, efficiency at full rigor. |
| Bishop, *Pattern Recognition and Machine Learning* | Sections 1.2, 3.1–3.4, 9.3–9.4 | Likelihood, regularization as priors, EM. |
| Murphy, *Probabilistic Machine Learning: An Introduction* | Chapters 4, 11 | MLE/MAP for standard models, ridge and lasso. |
| Gelman et al., *Bayesian Data Analysis* (3rd ed.) | Chapters 2, 4 | Priors, normal approximation to the posterior, MAP caveats. |
| Goodfellow, Bengio & Courville, *Deep Learning* | Chapter 5 | Maximum likelihood as the origin of ML loss functions. |
| Amari, *Information Geometry and Its Applications* | Chapters 1–3 | Fisher metric, natural gradient, exponential families. |

**Reading path**: Wasserman Ch. 9 for a fast rigorous pass, Casella & Berger Ch. 7 and 10 for the full estimation theory including Cramér–Rao, Bishop §3 and §9 for the prior-as-regularizer and EM viewpoints, then van der Vaart Ch. 5 for the asymptotics done properly and Amari for the geometric picture behind natural gradient.